# Modelling — LightGBM
## Credit Risk Scorecard Project

This notebook builds a LightGBM gradient boosting model.
We use all 274 features (vs 12 for the scorecard) and compare
performance against the WOE scorecard from notebook 03.

LightGBM trades interpretability for raw predictive power.
The comparison between the two models is the centrepiece of this project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import warnings
import os
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)

print("Libraries loaded successfully")
print(f"LightGBM version: {lgb.__version__}")

In [ ]:
# Load cleaned dataset
app = pd.read_csv('data/processed/app_clean.csv')

X = app.drop(columns=['TARGET', 'SK_ID_CURR'])
y = app['TARGET']

# Same 80/20 split as the scorecard notebook (test set is identical)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Carve a validation set OUT OF TRAIN, used only for early stopping
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=42, stratify=y_trainval
)

# Impute with TRAIN medians only
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_val = X_val.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"Default rate: {y_train.mean():.2%}")

In [ ]:
import re

# Clean column names - remove special characters
for df_ in (X_train, X_val, X_test):
    df_.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in df_.columns]

print("Column names cleaned")

# Handle class imbalance with scale_pos_weight
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos
print(f"Class imbalance ratio: {scale:.1f}")

# LightGBM parameters
params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'scale_pos_weight': scale,
    'random_state': 42,
    'verbose': -1
}

# Early stopping uses the VALIDATION set (carved from train), never the test set
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

print("Training LightGBM model...")

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

print(f"\nBest iteration: {model.best_iteration}")

In [ ]:
# Same 12 features as the scorecard (from notebook 03, after IV filter)
same_12 = ['EXT_SOURCE_3', 'EXT_SOURCE_2', 'EXT_SOURCE_1', 'DAYS_EMPLOYED',
           'AGE_YEARS', 'EMPLOYED_TO_AGE_RATIO', 'CREDIT_GOODS_RATIO',
           'AMT_CREDIT', 'PREV_REFUSED_COUNT', 'BUREAU_ACTIVE_COUNT',
           'AMT_ANNUITY', 'BUREAU_DEBT_MEAN']

d12 = lgb.Dataset(X_train[same_12], label=y_train)
v12 = lgb.Dataset(X_val[same_12], label=y_val, reference=d12)   # validation, not test
m12 = lgb.train(params, d12, num_boost_round=500, valid_sets=[v12],
                callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

# Final score on the untouched TEST set
auc_12 = roc_auc_score(y_test, m12.predict(X_test[same_12], num_iteration=m12.best_iteration))
print(f"Best iteration (12 features): {m12.best_iteration}")
print(f"LightGBM on same 12 features: Test AUC = {auc_12:.4f}")

In [ ]:
# Get predictions on test set
test_pred_lgb = model.predict(X_test, num_iteration=model.best_iteration)
test_auc_lgb = roc_auc_score(y_test, test_pred_lgb)

print(f"LightGBM Test AUC: {test_auc_lgb:.4f}")
print(f"Scorecard Test AUC: 0.7345 (from notebook 03)")
print(f"Improvement: +{(test_auc_lgb - 0.7345):.4f}")

# Feature importance
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print("\nTop 20 most important features:")
print(importance.head(20).to_string(index=False))

In [ ]:
# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 8))
top20 = importance.head(20)
ax.barh(top20['feature'], top20['importance'], color='steelblue')
ax.invert_yaxis()
ax.set_title('LightGBM Top 20 Feature Importance (by Gain)')
ax.set_xlabel('Importance (Gain)')
plt.tight_layout()
plt.savefig('outputs/04_feature_importance.png', dpi=150)
plt.show()

print("Feature importance chart saved ")


### Feature Importance Analysis

**External credit scores dominate.** EXT_SOURCE_3, EXT_SOURCE_2 and EXT_SOURCE_1
are the three strongest predictors by a wide margin (gain roughly 255k, 217k and
87k, against about 42k for the fourth). These are pre-computed creditworthiness
scores from external bureaus, effectively another institution's risk assessment.
Their dominance matches the WOE scorecard's IV ranking (EXT_SOURCE_3 and
EXT_SOURCE_2 are its top two features), so both models independently find the
same core signal.

**Engineered features add value.** Two ratios built during data preparation
appear in the top 20: CREDIT_GOODS_RATIO (#4) and ANNUITY_INCOME_RATIO (#17).
This suggests the domain-driven feature engineering contributes genuine signal
rather than noise. Note that EMPLOYED_TO_AGE_RATIO, which the scorecard uses, is
not in the LightGBM top 20, so the two models do not rank every feature alike.

**Behavioural history matters.** PREV_REFUSED_COUNT (#13) and several BUREAU_
aggregations (BUREAU_DAYS_CREDIT_MEAN, BUREAU_DEBT_MEAN, BUREAU_CREDIT_MEAN,
BUREAU_CREDIT_MAX) and PREV_ aggregations (PREV_DOWN_PAYMENT_MEAN,
PREV_DAYS_DECISION_MIN) rank in the top 20. This supports the effort of
aggregating and joining the external history tables.

**Fairness concern flagged.** CODE_GENDER_F ranks #11, so this model uses gender
as a predictor. Using gender in credit decisions is prohibited under fair-lending
rules in many jurisdictions. The model evaluated in notebooks 05 and 06 therefore
excludes all gender features, and notebook 05 checks how much predictive power
that costs.